<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/Module_01_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 01 Lab: Empirical Analysis of Foundation Models
**COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

This notebook follows the Module 01 lab manual. Run the cells **from top to bottom, in order**.

**Before you start:** your NRP API key must be saved in Colab Secrets (🔑 icon on the left) as `NRP_API_KEY`, with notebook access turned on. See Module 00.

| Model name | Developer and model | Thinking |
|---|---|---|
| `gpt-oss` | OpenAI gpt-oss-120b | low (cannot be turned off) |
| `qwen3` | Alibaba Qwen3.8 | none (off) |
| `gemma` | Google Gemma 4 31B | none (off) |

## Setup
### Step 1: Install libraries and connect to the NRP

In [ ]:
# Install necessary libraries
%pip install -q -U openai pandas tiktoken

import json
import time
import pandas as pd
import tiktoken
from openai import OpenAI, BadRequestError
from google.colab import userdata

# Secure API Initialization
# Before running, click the 'Key' icon in Colab and add NRP_API_KEY (see Module 00)
client = OpenAI(
    api_key=userdata.get("NRP_API_KEY").strip(),
    base_url="https://ellm.nrp-nautilus.io/v1",
)

### Step 2: Choose the models and the tokenizer

In [ ]:
# Models under test, with a "thinking" setting (reasoning_effort) valid for each.
MODELS = {
    "gpt-oss": "low",    # OpenAI gpt-oss-120b; thinking cannot be turned off
    "qwen3":   "none",   # Alibaba Qwen3.8; thinking turned off
    "gemma":   "none",   # Google Gemma 4 31B; thinking turned off
}

# Counts the tokens in the visible answer. o200k_base matches gpt-oss;
# Qwen and Gemma use their own tokenizers, so treat their counts as estimates.
encoder = tiktoken.get_encoding("o200k_base")

### Step 3: The experiment function
Runs one prompt on one model and returns latency, token counts, word count and the response. It retries if the shared NRP service is briefly busy.

In [ ]:
def run_experiment(model_name, prompt, temp=0.7, retries=3):
    """
    Standardized function to capture engineering metrics from the API call.
    Retries temporary failures on the shared NRP service.
    """
    for attempt in range(1, retries + 1):
        try:
            start_time = time.time()
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name,
                temperature=temp,
                max_tokens=2000,   # room for hidden thinking + the answer
                extra_body={"reasoning_effort": MODELS[model_name]},
            )
            latency = time.time() - start_time
            text = response.choices[0].message.content or ""

            return {
                "Model": model_name,
                "Thinking": MODELS[model_name],
                "Temp": temp,
                "Latency (s)": round(latency, 2),
                "Tokens": response.usage.total_tokens,
                "Completion Tokens": response.usage.completion_tokens,
                "Visible Tokens": len(encoder.encode(text)),
                "Word Count": len(text.split()),
                "Response": text
            }
        except BadRequestError as e:
            print(f"Invalid request for {model_name}: {e}")   # retrying will not help
            return None
        except Exception as e:
            print(f"Error with {model_name} (attempt {attempt}): {e}")
            time.sleep(10 * attempt)
    return None

### Step 4: JSON checker (used in Part 2)

In [ ]:
# JSON check for Part 2: the raw output must parse and contain all three fields.
def check_json(text):
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return False
    return all(field in data for field in ["definition", "advantages", "limitations"])

### Connection check
This should print the models available right now, and confirm the three models used in this lab are among them.

In [ ]:
available = [m.id for m in client.models.list().data]
print("Available models:", available)
for model_name in MODELS:
    print(f"{model_name}: {'OK' if model_name in available else 'NOT AVAILABLE - check the model list'}")

Available models: ['deepseek-v4-flash', 'qwen3-mc', 'gemma-mc', 'gpt-oss', 'gemma', 'gemma-vl', 'kimi', 'minimax', 'minimax-m2', 'agents', 'coding', 'general', 'cuda', 'kubernetes', 'long-context', 'reasoning', 'vision', 'gemma-small', 'gemma4-small', 'gemma4-12b', 'service', 'qwen3', 'qwen3-vl', 'qwen3-embedding', 'qwen3-small', 'gemma-small-e4b', 'glm-5']
gpt-oss: OK
qwen3: OK
gemma: OK


## Part 1: Determinism & Variability
**Prompt:** "Explain how blockchain works in exactly 100 words."

Each model runs the prompt **3 times at Temperature 0.7** and **3 times at Temperature 0**. This takes a few minutes.

In [ ]:
part1_prompt = "Explain how blockchain works in exactly 100 words."
part1_results = []

for model_name in MODELS:
    for temp in [0.7, 0]:
        for run in range(1, 4):
            result = run_experiment(model_name, part1_prompt, temp=temp)
            if result:
                result["Part"] = "Part 1"
                result["Run"] = run
                part1_results.append(result)
                print(f"{model_name} | temp={temp} | run {run} | {result['Latency (s)']} s | {result['Word Count']} words")
            time.sleep(2)   # share the NRP service fairly with other users

part1_df = pd.DataFrame(part1_results)
part1_df[["Model", "Thinking", "Temp", "Run", "Latency (s)", "Tokens", "Word Count"]]

gpt-oss | temp=0.7 | run 1 | 6.71 s | 100 words
gpt-oss | temp=0.7 | run 2 | 9.69 s | 100 words
gpt-oss | temp=0.7 | run 3 | 8.9 s | 100 words
gpt-oss | temp=0 | run 1 | 10.88 s | 100 words
gpt-oss | temp=0 | run 2 | 8.13 s | 100 words
gpt-oss | temp=0 | run 3 | 8.31 s | 100 words
qwen3 | temp=0.7 | run 1 | 1.91 s | 102 words
qwen3 | temp=0.7 | run 2 | 1.81 s | 95 words
qwen3 | temp=0.7 | run 3 | 1.9 s | 100 words
qwen3 | temp=0 | run 1 | 2.11 s | 96 words
qwen3 | temp=0 | run 2 | 1.95 s | 106 words
qwen3 | temp=0 | run 3 | 1.93 s | 106 words
gemma | temp=0.7 | run 1 | 2.48 s | 97 words
gemma | temp=0.7 | run 2 | 2.57 s | 98 words
gemma | temp=0.7 | run 3 | 2.48 s | 96 words
gemma | temp=0 | run 1 | 2.53 s | 96 words
gemma | temp=0 | run 2 | 2.54 s | 96 words
gemma | temp=0 | run 3 | 2.53 s | 96 words


,Model,Thinking,Temp,Run,Latency (s),Tokens,Word Count
0,gpt-oss,low,0.7,1,6.71,695,100
1,gpt-oss,low,0.7,2,9.69,917,100
2,gpt-oss,low,0.7,3,8.90,704,100
3,gpt-oss,low,0.0,1,10.88,693,100
4,gpt-oss,low,0.0,2,8.13,648,100
5,gpt-oss,low,0.0,3,8.31,689,100
6,qwen3,none,0.7,1,1.91,150,102
7,qwen3,none,0.7,2,1.81,139,95
8,qwen3,none,0.7,3,1.90,151,100
9,qwen3,none,0.0,1,2.11,148,96


**Look at the responses:** compare the wording of the three runs for each model and temperature.

In [ ]:
for (model_name, temp), group in part1_df.groupby(["Model", "Temp"]):
    print(f"===== {model_name} | Temperature {temp} =====")
    for _, row in group.iterrows():
        print(f"--- Run {row['Run']} ({row['Word Count']} words) ---")
        print(row["Response"][:600], "\n")

===== gemma | Temperature 0.0 =====
--- Run 1 (96 words) ---
Blockchain is a decentralized digital ledger that records transactions across a network of computers. Data is stored in "blocks," which are linked chronologically using cryptography to form a "chain." Each block contains a unique hash and the hash of the previous block, making the history immutable; changing one block requires altering all subsequent ones. Instead of a central authority, a consensus mechanism—like Proof of Work—validates new entries. Once confirmed, the data is permanent and transparent to all participants. This eliminates the need for intermediaries, ensuring security, trust, 

--- Run 2 (96 words) ---
Blockchain is a decentralized digital ledger that records transactions across a network of computers. Data is stored in "blocks," which are linked chronologically using cryptography to form a "chain." Each block contains a unique hash and the hash of the previous block, making the history immutable; changing o

## Part 2: Structured Output Reliability
**Prompt:** "Return a JSON object with fields: 'definition', 'advantages', and 'limitations' explaining cloud computing."

Each model runs the prompt **3 times at Temperature 0**. `check_json()` marks a response valid only if the raw text parses as JSON and contains all three fields.

In [ ]:
part2_prompt = ("Return a JSON object with fields: 'definition', 'advantages', "
                "and 'limitations' explaining cloud computing.")
part2_results = []

for model_name in MODELS:
    for run in range(1, 4):
        result = run_experiment(model_name, part2_prompt, temp=0)
        if result:
            result["Part"] = "Part 2"
            result["Run"] = run
            result["JSON Valid"] = check_json(result["Response"])
            result["Has Code Fence"] = "```" in result["Response"]
            part2_results.append(result)
            print(f"{model_name} | run {run} | JSON valid: {result['JSON Valid']}")
        time.sleep(2)

part2_df = pd.DataFrame(part2_results)
part2_df[["Model", "Run", "JSON Valid", "Has Code Fence", "Latency (s)", "Tokens"]]

gpt-oss | run 1 | JSON valid: False
gpt-oss | run 2 | JSON valid: False
gpt-oss | run 3 | JSON valid: False
qwen3 | run 1 | JSON valid: True
qwen3 | run 2 | JSON valid: True
qwen3 | run 3 | JSON valid: True
gemma | run 1 | JSON valid: False
gemma | run 2 | JSON valid: False
gemma | run 3 | JSON valid: False


,Model,Run,JSON Valid,Has Code Fence,Latency (s),Tokens
0,gpt-oss,1,False,True,4.01,572
1,gpt-oss,2,False,True,3.29,499
2,gpt-oss,3,False,True,3.29,499
3,qwen3,1,True,False,3.93,351
4,qwen3,2,True,False,4.22,393
5,qwen3,3,True,False,3.84,343
6,gemma,1,False,True,4.06,255
7,gemma,2,False,True,4.02,255
8,gemma,3,False,True,4.07,255


**Inspect any failures:** print the raw output of responses that failed the check.

In [ ]:
for _, row in part2_df[~part2_df["JSON Valid"]].iterrows():
    print(f"===== {row['Model']} | run {row['Run']} =====")
    print(row["Response"][:800], "\n")
if part2_df["JSON Valid"].all():
    print("All responses were valid JSON.")

===== gpt-oss | run 1 =====
```json
{
  "definition": "Cloud computing is the delivery of computing resources—such as servers, storage, databases, networking, software, analytics, and intelligence—over the internet (the “cloud”) on a pay‑as‑you‑go basis. Instead of owning and managing physical hardware and software on‑premises, users can provision and scale resources dynamically from cloud service providers.",
  "advantages": [
    "Scalability: Resources can be rapidly increased or decreased to match workload demand, enabling elastic performance without upfront capacity planning.",
    "Cost efficiency: Pay‑only-for‑what‑you‑use models eliminate large capital expenditures and reduce operational costs related to hardware maintenance, power, and cooling.",
    "Speed & agility: New environments, applications, and serv 

===== gpt-oss | run 2 =====
```json
{
  "definition": "Cloud computing is the delivery of computing resources—such as servers, storage, databases, networking, software, 

## Part 3: Hallucination Test
**Prompt:** "Who won the Nobel Prize in Physics in 2026?"

The 2026 prize was announced in October 2026, after these models were trained, so a confident named winner is a hallucination.

In [ ]:
part3_prompt = "Who won the Nobel Prize in Physics in 2026?"
part3_results = []

for model_name in MODELS:
    result = run_experiment(model_name, part3_prompt, temp=0)
    if result:
        result["Part"] = "Part 3"
        result["Run"] = 1
        part3_results.append(result)
        print(f"===== {model_name} =====")
        print(result["Response"], "\n")
    time.sleep(2)

part3_df = pd.DataFrame(part3_results)

===== gpt-oss =====
I’m afraid I don’t have that information. The Nobel Prizes for 2026 haven’t been announced yet (and, as of today, the year 2026 is still in the future), so the laureates are not known. Once the Royal Swedish Academy of Sciences makes the official announcement—typically in early October each year—the winner(s) will be publicly listed on the Nobel Prize website and in the news. If you check back after the announcement, you’ll be able to find the name(s) of the 2026 Physics laureate(s) there. 

===== qwen3 =====
It is not possible to state who won the Nobel Prize in Physics in 2026 because **that year has not yet occurred**.

The Nobel Prizes are awarded annually, and the 2026 laureates will not be announced until **October 2026**. As of today, the most recent Nobel Prize in Physics was awarded in **2024**.

For reference, the **2024 Nobel Prize in Physics** was awarded to:
*   **John J. Hopfield** and **Geoffrey E. Hinton** "for foundational discoveries and inventions

**Label each model's answer yourself.** Read the responses above, then fill in one label per model:
- `Admitted cutoff`: said it does not know or cited a knowledge cutoff
- `Fabricated winner`: confidently named a winner
- `Hedged`: guessed, but flagged uncertainty

In [ ]:
HALLUCINATION = {
    "gpt-oss": "",   # e.g. "Admitted cutoff"
    "qwen3":   "",
    "gemma":   "",
}
print(HALLUCINATION)

{'gpt-oss': '', 'qwen3': '', 'gemma': ''}


## Section 4: Required Analysis Table
Combines all three parts into the summary table required in the lab manual.

In [ ]:
summary = (part1_df.groupby(["Model", "Thinking", "Temp"])
           .agg(**{"Avg Latency (s)": ("Latency (s)", "mean"),
                   "Avg Tokens": ("Tokens", "mean"),
                   "Avg Word Count": ("Word Count", "mean")})
           .round(2)
           .reset_index())

json_rate = part2_df.groupby("Model")["JSON Valid"].mean().mul(100).round(0)
summary["JSON Validity (%)"] = summary["Model"].map(json_rate)
summary["Hallucination"] = summary["Model"].map(HALLUCINATION)
summary

,Model,Thinking,Temp,Avg Latency (s),Avg Tokens,Avg Word Count,JSON Validity (%),Hallucination
0,gemma,none,0.0,2.53,144.00,96.00,0.0,
1,gemma,none,0.7,2.51,141.00,97.00,0.0,
2,gpt-oss,low,0.0,9.11,676.67,100.00,0.0,
3,gpt-oss,low,0.7,8.43,772.00,100.00,0.0,
4,qwen3,none,0.0,2.00,152.00,102.67,100.0,
5,qwen3,none,0.7,1.87,146.67,99.00,100.0,


## Hidden thinking tokens (for the report)
`Completion Tokens` includes hidden thinking; `Visible Tokens` counts only the answer the user sees. The difference shows how much of each model's usage the user never sees.

In [ ]:
thinking = (part1_df.groupby("Model")[["Completion Tokens", "Visible Tokens"]].mean().round(0))
thinking["Hidden (approx.)"] = thinking["Completion Tokens"] - thinking["Visible Tokens"]
thinking

,Completion Tokens,Visible Tokens,Hidden (approx.)
Model,,,
gemma,118.0,118.0,0.0
gpt-oss,647.0,129.0,518.0
qwen3,124.0,124.0,0.0


## Save your results
Saves every run and the summary table. Download both files from the folder icon on the left (⋮ → Download).

In [ ]:
all_results = pd.concat([part1_df, part2_df, part3_df], ignore_index=True)
all_results.to_csv("module01_results.csv", index=False)
summary.to_csv("module01_summary.csv", index=False)
print("Saved module01_results.csv and module01_summary.csv")

Saved module01_results.csv and module01_summary.csv


## Before you submit
- [ ] All cells ran from top to bottom without errors.
- [ ] The `HALLUCINATION` labels are filled in.
- [ ] `module01_results.csv` and `module01_summary.csv` are saved.
- [ ] This notebook does not contain your API key (it is read from Colab Secrets).
- [ ] Submit this notebook (Share link or .ipynb) with your Technical Report.